# Bitcoin OTC: the data and the network

Each row of the SNAP file is one rating: **source rates target**, with a score from −10 to +10 and a Unix time. A rating is an opinion, not a verified trade. In this notebook we check the file and build the network used in the rest of the project.

## 1. Read the file

We keep the downloaded file unchanged and print its SHA-256 checksum, so everyone in the team can check they have the same data. It should start with `6424ac98`.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import networkx as nx

ROOT = Path('..')  # this notebook is in notebooks/, the project folder is one level up
DATA = ROOT / 'data' / 'soc-sign-bitcoinotc.csv.gz'

print('SHA-256:', hashlib.sha256(DATA.read_bytes()).hexdigest())

ratings = pd.read_csv(DATA, header=None, names=['source', 'target', 'rating', 'time'])
ratings['date'] = pd.to_datetime(ratings['time'], unit='s', utc=True)
print('Rows:', len(ratings))
ratings.head()

SHA-256: 6424ac981dad3a019f697fc1b9fcd85c19d8d9f039797758e9ffb6fea100c373
Rows: 35592


,source,target,rating,time,date
0,6,2,4,1.289242e+09,2010-11-08 18:45:11.728359938+00:00
1,6,5,2,1.289242e+09,2010-11-08 18:45:41.533780098+00:00
2,1,15,1,1.289243e+09,2010-11-08 19:05:40.390490055+00:00
3,4,3,7,1.289245e+09,2010-11-08 19:41:17.369750023+00:00
4,13,16,8,1.289254e+09,2010-11-08 22:10:54.447459936+00:00


## 2. Check the data before using it

We look for missing values, users rating themselves, repeated source–target pairs, the number of positive and negative ratings, and the time span. If any of these were a problem we would have to say how we handled it.

In [2]:
users = set(ratings['source']) | set(ratings['target'])

print('Users:', len(users))
print('Missing cells:', ratings.isna().sum().sum())
print('Self-ratings:', (ratings['source'] == ratings['target']).sum())
print('Repeated source-target pairs:', ratings.duplicated(['source', 'target']).sum())
print('Positive ratings:', (ratings['rating'] > 0).sum())
print('Negative ratings:', (ratings['rating'] < 0).sum())
print('Zero ratings:', (ratings['rating'] == 0).sum())
print('First rating:', ratings['date'].min())
print('Last rating:', ratings['date'].max())

Users: 5881
Missing cells: 0
Self-ratings: 0
Repeated source-target pairs: 0
Positive ratings: 32029
Negative ratings: 3563
Zero ratings: 0
First rating: 2010-11-08 18:45:11.728359938+00:00
Last rating: 2016-01-25 01:12:03.757280111+00:00


In [3]:
# positive (True) and negative (False) ratings per year
ratings['year'] = ratings['date'].dt.year
ratings['positive'] = ratings['rating'] > 0
pd.crosstab(ratings['year'], ratings['positive'])

positive,False,True
year,,
2010,0,142
2011,155,7603
2012,810,8622
2013,1844,11138
2014,670,3555
2015,82,929
2016,2,40


## 3. Build the network

Nodes are users. A directed link $u \to v$ means user $u$ gave user $v$ a **positive** rating (score above 0); the score is kept as a link attribute. We add all users as nodes first, so users who only appear in negative ratings stay in the network as isolated nodes instead of disappearing.

The **in-degree** $k^{in}$ of a user is the number of positive ratings they received, and the **out-degree** $k^{out}$ the number they gave. In a directed network the average in-degree is $L/N$.

In [4]:
positive = ratings[ratings['rating'] > 0]

DG = nx.from_pandas_edgelist(positive, 'source', 'target', edge_attr='rating', create_using=nx.DiGraph)
DG.add_nodes_from(users)

N = DG.number_of_nodes()
L = DG.number_of_edges()
print('N =', N, ' L =', L)
print('Average in-degree L/N =', round(L / N, 3))
print('Weakly connected components:', nx.number_weakly_connected_components(DG))

N = 5881  L = 32029
Average in-degree L/N = 5.446
Weakly connected components: 317


## 4. Time

Every rating has a timestamp, so we can follow the network as it grows. The network "at the end of 2013" contains all ratings recorded up to then. Ratings never expire in the file, so this is the recorded history, not necessarily who still trusted whom at that date. Notebook 03 uses this for our focused question.